# PDCP · N11 — Explanation faithfulness: what models say vs what drives their choices
**Notebook 11 of 12.** Kaggle: Accelerator **None (CPU)** · Internet **On** · **Save & Run All (Commit)**. Expected ≈ 10–20 min.

**Inputs:** `kbs-n3-output` (pairs, drawings) · `kbs-n5-output` (E1) · `kbs-n6-part2` (E2) · `kbs-no8-output` (E3) · `kbs-n10-output` (E4) · `kbs-n11labels` (300 human-labelled explanations, blind to model and answer).

1. **Coder validation.** A rule-based coder marks five cue categories in each explanation: CROSSINGS, EDGE_LENGTH, NODE_SPREAD, PATH_DISTANCE (the actual definition of stress), TIDY_GENERIC. Rules were tuned on the odd-numbered half of the human labels and frozen; agreement is reported separately on the held-out even half (Cohen's κ, precision, recall).
2. **Mention rates** of each category by model, experiment, prompt and rendering.
3. **Said vs used.** How often a model *names* a cue (E1) against how strongly that cue *causally* moves its choices (E2 counterfactual edits with stress fixed, corrected by the null-edit floor) and against its standardized weight in E1 choices.
4. **Counterfactual faithfulness (CCT, Siegel et al. 2024).** For every counterfactual pair: did the edit change the answer (vs the base pair, same model, same order), and does the explanation of the edited pair mention the edited cue? CCT = (mean change | mentioned − mean change | not mentioned) / sd(change); > 0 means explanations name the cue when it matters. 95 % CI by bootstrap over base pairs.
5. **Claim validity.** When an explanation mentions crossings, does the chosen drawing really have fewer crossings? When it mentions path distances, is the choice more often correct?

**Outputs** (`/kaggle/working/n11_output/`): `coded_reasons.parquet`, `n11_coder_validation.csv`, `n11_mention_rates.csv`, `n11_said_vs_used.csv`, `n11_cct.csv`, `n11_claim_validity.csv`, `n11_report.json`, figures `fig_n11_*.png`, `coder.py`, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS ====
N03_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output"
N05_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n5-output"
N06_ROOTS = ["/kaggle/input/datasets/resadussalehinrafsan/kbs-n6-part2",
             "/kaggle/input/datasets/resadussalehinrafsan/kbs-n06-part1-output"]     # second one optional (skipped if not attached)
N08_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-no8-output"
N10_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n10-output"
LABELS_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n11labels"
OUTPUT_DIR = "/kaggle/working/n11_output"
N_BOOT = 2000
SEED = 2026
CUE2CAT = {"nu": ["NODE_SPREAD"], "eld": ["EDGE_LENGTH"], "xr": ["CROSSINGS"], "mel": ["EDGE_LENGTH", "NODE_SPREAD"]}   # edited cue -> explanation category
N_CPU = 4

In [2]:
import os, sys, json, time, glob, shutil, re
T0 = time.time(); os.makedirs(OUTPUT_DIR, exist_ok=True)
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score
def find(root, name, required=True):
    f = sorted(glob.glob(f"{root}/**/{name}", recursive=True))
    if not f:
        assert not required, f"{name} not found under {root}"; return None
    return os.path.dirname(f[0])
N03 = find(N03_ROOT, "pairs.parquet"); N06 = find(N06_ROOTS[0], "mel_cf_pairs.parquet")
shutil.copy(f"{N06}/pdcp_utils.py", f"{OUTPUT_DIR}/pdcp_utils.py"); sys.path.insert(0, OUTPUT_DIR); import pdcp_utils as U
PR = pd.read_parquet(f"{N03}/pairs.parquet"); DR = pd.read_parquet(f"{N03}/drawings.parquet"); MCP = pd.read_parquet(f"{N06}/mel_cf_pairs.parquet")
LAB = sorted(glob.glob(f"{LABELS_ROOT}/**/*.xlsx", recursive=True)); assert LAB, "label workbook not found"
print("N03:", N03, "| N06:", N06, "| labels:", LAB[0])

N03: /kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output/n03_output | N06: /kaggle/input/datasets/resadussalehinrafsan/kbs-n6-part2/n06_output | labels: /kaggle/input/datasets/resadussalehinrafsan/kbs-n11labels/N11_reason_labels (1).xlsx


## 1. The coder (frozen) and its validation against the human labels

In [3]:
%%writefile /kaggle/working/n11_output/coder.py
import re
E = r"(?:edges?|lines?|connections?|links?)"
RULES = {
 "CROSSINGS": [r"cross", r"intersect", r"tangl", r"overlap\w*\s+(?:of\s+)?" + E, E + r"\s+(?:\w+\s+){0,2}overlap", r"\boverlapping\b", r"knot"],
 "EDGE_LENGTH": [r"\b(?:long|short|longer|shorter|lengthy|stretched|uniform|equal|similar|consistent|varying|variable)\w*\s+(?:\w+\s+){0,2}" + E,
                 E + r"\s+(?:\w+\s+){0,3}(?:long|short|longer|shorter|length|stretched)", r"edge[- ]lengths?", r"lengths? of (?:the )?" + E],
 "NODE_SPREAD": [r"compact", r"cluster", r"spread", r"dispers", r"\bdense", r"crowd", r"spac(?:ing|ed)", r"packed", r"scatter", r"separat",
                 r"evenly", r"distribut", r"close together", r"closer together", r"concentrat", r"bunch", r"sparse", r"clump", r"gaps?\b", r"isolated", r"grouped", r"centrali[sz]", r"close(?:r)? to each other"],
 "PATH_DISTANCE": [r"graph[- ]?theoretic", r"graph distance", r"\bpaths?\b", r"shortest", r"edges? apart", r"\bhops?\b", r"proportional",
                   r"neighbo", r"(?:their|actual|true|real|graph|structural|topolog\w*)\s+(?:\w+\s+)?distances?", r"distances?\s+(?:\w+\s+){0,3}(?:match|align|reflect|preserv|correspond|consistent)",
                   r"(?:match|align|reflect|preserv|correspond)\w*\s+(?:\w+\s+){0,3}distances?", r"according to (?:their )?connections", r"connected nodes", r"in the graph", r"based on (?:their )?connections"],
 "TIDY_GENERIC": [r"organi[sz]", r"\bclear(?:er)?\b(?!\s+separation)", r"\bclean", r"mess", r"clutter", r"structured", r"balanc", r"symmetr", r"chao", r"\bneat", r"tidy",
                  r"orderly", r"readab", r"confus", r"coheren", r"aesthetic", r"complex"],
}
CATS = list(RULES)
_C = {k: [re.compile(p, re.I) for p in v] for k, v in RULES.items()}
def code(text):
    t = str(text or "")
    return {k: int(any(p.search(t) for p in v)) for k, v in _C.items()}


Writing /kaggle/working/n11_output/coder.py


In [4]:
import importlib; import coder as CD; importlib.reload(CD); CATS = CD.CATS
H = pd.read_excel(LAB[0], "Label"); H["n"] = H.id.str[1:].astype(int)
H[CATS] = H[CATS].fillna(0).astype(int)
KEY = pd.read_excel(LAB[0], "key")
Kc = pd.DataFrame([CD.code(t) for t in H.iloc[:, 1]])
val = []
for part, m in [("tuning half (odd ids)", H.n % 2 == 1), ("held-out half (even ids)", H.n % 2 == 0), ("all 300", H.n > 0)]:
    for c in CATS:
        y, k = H.loc[m, c], Kc.loc[m, c]
        val.append({"part": part, "category": c, "n": int(m.sum()), "human_pos": int(y.sum()), "coder_pos": int(k.sum()),
                    "agreement": round(float((y == k).mean()), 3), "kappa": round(float(cohen_kappa_score(y, k)), 3),
                    "precision": round(float(precision_score(y, k, zero_division=0)), 3), "recall": round(float(recall_score(y, k, zero_division=0)), 3)})
VAL = pd.DataFrame(val); VAL.to_csv(f"{OUTPUT_DIR}/n11_coder_validation.csv", index=False)
print(VAL[VAL.part.str.startswith("held")].to_string(index=False))
hm = H.merge(KEY[["id", "model", "exp"]], on="id")
print("human mention rates by model:"); print(hm.groupby("model")[CATS].mean().round(2).to_string())

                    part      category   n  human_pos  coder_pos  agreement  kappa  precision  recall
held-out half (even ids)     CROSSINGS 150         15         15      1.000  1.000      1.000   1.000
held-out half (even ids)   EDGE_LENGTH 150          4          5      0.980  0.656      0.600   0.750
held-out half (even ids)   NODE_SPREAD 150        123        116      0.940  0.815      0.991   0.935
held-out half (even ids) PATH_DISTANCE 150         97         90      0.940  0.873      0.989   0.918
held-out half (even ids)  TIDY_GENERIC 150         51         55      0.947  0.883      0.891   0.961
human mention rates by model:
             CROSSINGS  EDGE_LENGTH  NODE_SPREAD  PATH_DISTANCE  TIDY_GENERIC
model                                                                        
InternVL-4B       0.15         0.01         0.81           0.39          0.53
Q25VL-7B          0.17         0.01         0.84           0.51          0.11
Q3VL-2B           0.08         0.05         0.

## 2. All answers with explanations (E1–E4)

In [5]:
def load(roots, sub, exp, default_prompt="P-Def"):
    files = []
    for r in roots: files += glob.glob(f"{r}/**/results/{sub}/*.jsonl", recursive=True)
    rows = []
    for f in files:
        for l in open(f):
            try: r = json.loads(l)
            except Exception: continue
            p = r.get("parsed") or {}; reason = p.get("reason")
            if not reason:
                m = re.search(r'"reason"\s*:\s*"([^"]*)', r.get("raw_text") or ""); reason = m.group(1) if m else None
            rows.append({"exp": exp, "model": r["model"], "prompt_id": r.get("prompt_id") or default_prompt, "pair_id": r["pair_id"], "order": r["order"],
                         "ans": p.get("answer"), "confidence": p.get("confidence"), "reason": reason})
    return pd.DataFrame(rows)
A = pd.concat([load([N05_ROOT], "e1", "E1"), load(N06_ROOTS, "e2", "E2"), load([N08_ROOT], "e3", "E3"), load([N10_ROOT], "e4", "E4")], ignore_index=True)
A = A.drop_duplicates(["exp", "model", "prompt_id", "pair_id", "order"], keep="last").reset_index(drop=True)
A["choice"] = np.where(A.ans == "same", "same", np.where(A.ans.isin(["1", "2"]), np.where((A.ans == "1") == (A.order == "AB"), "A", "B"), None))
A["base_id"] = A.pair_id.str.split("__").str[0]; A["variant"] = np.where(A.pair_id.str.contains("__"), A.pair_id.str.split("__").str[1], "default")
u = A.reason.dropna().unique(); coded = dict(zip(u, [CD.code(t) for t in u]))
for c in CATS: A[c] = A.reason.map(lambda t: coded[t][c] if isinstance(t, str) else np.nan)
A["has_reason"] = A.reason.notna() & (A.reason.astype(str).str.len() > 3)
print(A.groupby(["exp", "model"]).agg(n=("pair_id", "size"), with_reason=("has_reason", "mean")).round(3).to_string())

                    n  with_reason
exp model                         
E1  InternVL-4B  4430        1.000
    Q25VL-7B     4430        1.000
    Q3VL-2B      4430        1.000
    Q3VL-4B      4430        1.000
E2  InternVL-4B  8770        1.000
    Q25VL-7B     8770        1.000
    Q3VL-2B      8770        1.000
    Q3VL-4B      8770        1.000
E3  InternVL-4B  5508        0.897
    Q25VL-7B     4304        0.907
    Q3VL-2B      5512        0.927
E4  InternVL-2B  1168        1.000
    InternVL-4B  4400        0.909
    InternVL-8B  1168        1.000
    Q25VL-3B     1168        1.000
    Q25VL-7B     3200        1.000
    Q3VL-2B      3600        1.000


## 3. Pair metadata (cue differences, counterfactual edits)

In [6]:
meta = pd.concat([PR, MCP], ignore_index=True).drop_duplicates("pair_id").set_index("pair_id")
mainp = PR[PR.set.isin(["main", "e1b_mooney"])]
Di = DR.set_index("drawing_id")
def mel_of(did):
    import sys; sys.path.insert(0, OUTPUT_DIR); import numpy as np, pdcp_utils as U
    r = Di.loc[did]; gc = U.GraphCache(U.make_graph(r.family, int(r.n), int(r.graph_seed))); return did, U.mean_edge_length(np.array(r.coords).reshape(-1, 2), gc.E)
dids = sorted(set(mainp.drawing_A) | set(mainp.drawing_B))
MEL = dict(Parallel(n_jobs=N_CPU, batch_size=64)(delayed(mel_of)(d) for d in dids))
mainp = mainp.assign(d_mel=mainp.drawing_B.map(MEL) - mainp.drawing_A.map(MEL)).set_index("pair_id")
A = A.join(meta[["cond", "dstress_design", "correct", "cf_proxy", "cf_delta", "base_pair", "d_ksm", "d_nu", "d_eld", "d_xr"]], on="base_id")
A["is_correct"] = np.where(A.choice.notna(), A.choice == A.correct, np.nan)
print(f"{len(A)} answers | {A.has_reason.sum()} with explanation | {(time.time()-T0)/60:.1f} min")

82828 answers | 81059 with explanation | 0.5 min


## 4. Mention rates

In [7]:
R = A[A.has_reason]
MR = R.groupby(["exp", "model", "prompt_id", "variant"])[CATS].mean().round(4)
MR.insert(0, "n", R.groupby(["exp", "model", "prompt_id", "variant"]).size()); MR = MR.reset_index()
MR.to_csv(f"{OUTPUT_DIR}/n11_mention_rates.csv", index=False)
E1R = R[R.exp == "E1"]
print("E1 mention rates (P-Def):"); print(E1R.groupby("model")[CATS].mean().round(3).to_string())

E1 mention rates (P-Def):
             CROSSINGS  EDGE_LENGTH  NODE_SPREAD  PATH_DISTANCE  TIDY_GENERIC
model                                                                        
InternVL-4B      0.391        0.007        0.851          0.196         0.604
Q25VL-7B         0.005        0.000        0.860          0.902         0.000
Q3VL-2B          0.026        0.048        0.959          0.664         0.249
Q3VL-4B          0.118        0.000        0.849          0.985         0.283


## 5. Said vs used

In [8]:
def pa(df):   # order-averaged P(choose A) per (model, pair)
    d = df[df.choice.notna()].assign(a=lambda x: x.choice.map({"A": 1.0, "B": 0.0, "same": 0.5}))
    return d.groupby(["model", "pair_id"]).a.mean()
base = pa(A[(A.exp == "E1") & (A.prompt_id == "P-Def")])
cf = A[(A.exp == "E2") & A.cf_proxy.notna()]
pcf = pa(cf).rename("a_cf").reset_index().join(meta[["base_pair", "cf_proxy", "cf_delta"]], on="pair_id")
pcf["a_base"] = [base.get((m, b), np.nan) for m, b in zip(pcf.model, pcf.base_pair)]
pcf = pcf.dropna(subset=["a_base"]); pcf["abs_shift"] = (pcf.a_cf - pcf.a_base).abs()
null = pcf[pcf.cf_proxy == "null"].groupby("model").abs_shift.mean()
caus = pcf[pcf.cf_proxy != "null"].groupby(["model", "cf_proxy"]).abs_shift.mean().rename("abs_shift").reset_index()
caus["null_floor"] = caus.model.map(null); caus["causal_effect"] = caus.abs_shift - caus.null_floor.fillna(0)
# standardized weight of each cue on E1 choices (main pairs)
w = []
for m, s in base.groupby(level=0):
    s = s.droplevel(0); X = mainp.loc[mainp.index.intersection(s.index), ["d_ksm", "d_nu", "d_eld", "d_xr", "d_mel"]].dropna()
    if len(X) < 50: continue
    Z = (X - X.mean()) / X.std(); y = s[X.index]; y = (y - y.mean()) / (y.std() + 1e-12)
    b = np.linalg.lstsq(np.c_[np.ones(len(Z)), Z.values], y.values, rcond=None)[0][1:]
    w += [{"model": m, "cf_proxy": k[2:], "e1_weight": float(v)} for k, v in zip(X.columns, b) if k != "d_ksm"]
W = pd.DataFrame(w)
said = []
for m, g in E1R.groupby("model"):
    for cue, cats in CUE2CAT.items(): said.append({"model": m, "cf_proxy": cue, "mention_rate": float(g[cats].max(axis=1).mean())})
SVU = pd.DataFrame(said).merge(caus, on=["model", "cf_proxy"], how="left").merge(W, on=["model", "cf_proxy"], how="left")
SVU.round(4).to_csv(f"{OUTPUT_DIR}/n11_said_vs_used.csv", index=False); print(SVU.round(3).to_string(index=False))

      model cf_proxy  mention_rate  abs_shift  null_floor  causal_effect  e1_weight
InternVL-4B       nu         0.851      0.290       0.235          0.055     -0.219
InternVL-4B      eld         0.007      0.247       0.235          0.011     -0.059
InternVL-4B       xr         0.391      0.264       0.235          0.029      0.134
InternVL-4B      mel         0.854      0.273       0.235          0.037      0.127
   Q25VL-7B       nu         0.860      0.212       0.204          0.008      0.035
   Q25VL-7B      eld         0.000      0.205       0.204          0.001     -0.067
   Q25VL-7B       xr         0.005      0.197       0.204         -0.006      0.180
   Q25VL-7B      mel         0.860      0.233       0.204          0.029      0.086
    Q3VL-2B       nu         0.959      0.213       0.166          0.046     -0.201
    Q3VL-2B      eld         0.048      0.188       0.166          0.022      0.082
    Q3VL-2B       xr         0.026      0.197       0.166          0.031    

## 6. Counterfactual faithfulness (CCT)

In [9]:
b_ans = A[(A.exp == "E1") & (A.prompt_id == "P-Def")].set_index(["model", "pair_id", "order"]).choice
X = A[(A.exp == "E2") & A.cf_proxy.notna() & (A.cf_proxy != "null") & A.has_reason & A.choice.notna()].copy()
X["base_choice"] = [b_ans.get((m, b, o)) for m, b, o in zip(X.model, X.base_pair, X.order)]
X = X[X.base_choice.notna()]
X["changed"] = (X.choice != X.base_choice).astype(float)
X["mentions"] = [int(any(r[c] == 1 for c in CUE2CAT[p])) for r, p in zip(X[CATS].to_dict("records"), X.cf_proxy)]
def cct(d):
    a, b = d[d.mentions == 1].changed, d[d.mentions == 0].changed; sd = d.changed.std()
    return (a.mean() - b.mean()) / sd if len(a) and len(b) and sd > 0 else np.nan
rng = np.random.default_rng(SEED); rows = []
for (m, p), d in X.groupby(["model", "cf_proxy"]):
    bp = d.base_pair.unique(); grp = {k: g for k, g in d.groupby("base_pair")}; boots = []
    for _ in range(N_BOOT):
        s = pd.concat([grp[k] for k in rng.choice(bp, len(bp))]); boots.append(cct(s))
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    rows.append({"model": m, "cf_proxy": p, "n": len(d), "p_changed": round(d.changed.mean(), 3), "p_mention": round(d.mentions.mean(), 3),
                 "changed_if_mentioned": round(d[d.mentions == 1].changed.mean(), 3) if (d.mentions == 1).any() else np.nan,
                 "changed_if_not": round(d[d.mentions == 0].changed.mean(), 3), "cct": round(cct(d), 3), "ci_lo": round(lo, 3), "ci_hi": round(hi, 3)})
CCT = pd.DataFrame(rows); CCT.to_csv(f"{OUTPUT_DIR}/n11_cct.csv", index=False); print(CCT.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1437: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1437: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


      model cf_proxy    n  p_changed  p_mention  changed_if_mentioned  changed_if_not    cct  ci_lo  ci_hi
InternVL-4B      eld 2302      0.266      0.007                 0.235           0.266 -0.070 -0.482  0.459
InternVL-4B      mel 2302      0.299      0.877                 0.298           0.303 -0.010 -0.151  0.132
InternVL-4B       nu 1138      0.313      0.896                 0.301           0.415 -0.246 -0.489 -0.001
InternVL-4B       xr 2238      0.278      0.354                 0.238           0.300 -0.138 -0.241 -0.024
   Q25VL-7B      eld 2302      0.210      0.000                   NaN           0.210    NaN    NaN    NaN
   Q25VL-7B      mel 2302      0.241      0.849                 0.249           0.196  0.124 -0.020  0.258
   Q25VL-7B       nu 1138      0.219      0.870                 0.230           0.142  0.214  0.045  0.371
   Q25VL-7B       xr 2238      0.205      0.004                 0.250           0.204  0.113 -0.522  1.150
    Q3VL-2B      eld 2302      0.197 

## 7. Claim validity (E1)

In [10]:
E1m = E1R[E1R.choice.isin(["A", "B"])].join(mainp[["d_mel"]], on="base_id")
sgn = np.where(E1m.choice == "A", 1, -1)                     # d_* = B - A: positive -> chosen A has the lower value
E1m = E1m.assign(chosen_fewer_cross=(sgn * E1m.d_xr) > 0, chosen_more_uniform_len=(sgn * E1m.d_eld) > 0, chosen_shorter_edges=(sgn * E1m.d_mel) > 0)
cv = []
for m, g in E1m.groupby("model"):
    nz = g[g.dstress_design > 0]
    cv.append({"model": m,
               "crossing_claims": int(g.CROSSINGS.sum()), "crossing_claim_true": round(float(g[g.CROSSINGS == 1].chosen_fewer_cross.mean()), 3),
               "crossing_true_when_not_claimed": round(float(g[g.CROSSINGS == 0].chosen_fewer_cross.mean()), 3),
               "edge_length_claims": int(g.EDGE_LENGTH.sum()),
               "chosen_shorter_edges_all": round(float(g.chosen_shorter_edges.mean()), 3),
               "acc_if_path_mentioned": round(float(nz[nz.PATH_DISTANCE == 1].is_correct.mean()), 3),
               "acc_if_path_not_mentioned": round(float(nz[nz.PATH_DISTANCE == 0].is_correct.mean()), 3),
               "acc_if_tidy_only": round(float(nz[(nz.TIDY_GENERIC == 1) & (nz[["CROSSINGS", "EDGE_LENGTH", "NODE_SPREAD", "PATH_DISTANCE"]].sum(1) == 0)].is_correct.mean()), 3)})
CV = pd.DataFrame(cv); CV.to_csv(f"{OUTPUT_DIR}/n11_claim_validity.csv", index=False); print(CV.to_string(index=False))

      model  crossing_claims  crossing_claim_true  crossing_true_when_not_claimed  edge_length_claims  chosen_shorter_edges_all  acc_if_path_mentioned  acc_if_path_not_mentioned  acc_if_tidy_only
InternVL-4B             1732                0.477                           0.512                  32                     0.549                  0.419                      0.509             0.126
   Q25VL-7B               22                0.818                           0.533                   2                     0.485                  0.471                      0.556               NaN
    Q3VL-2B              114                0.491                           0.505                 214                     0.378                  0.314                      0.391             0.000
    Q3VL-4B              523                0.407                           0.534                   2                     0.398                  0.322                      0.194             1.000


## 8. Figures

In [11]:
short = {"CROSSINGS": "crossings", "EDGE_LENGTH": "edge length", "NODE_SPREAD": "node spread", "PATH_DISTANCE": "path distance", "TIDY_GENERIC": "tidy/generic"}
fig, ax = plt.subplots(1, 2, figsize=(15, 3.8), gridspec_kw={"width_ratios": [1.2, 1]})
hv = VAL[VAL.part.str.startswith("held")].set_index("category")
ax[0].bar([short[c] for c in CATS], hv.kappa.reindex(CATS)); ax[0].set_ylim(0, 1); ax[0].axhline(0.8, ls=":", c="grey")
for i, c in enumerate(CATS): ax[0].text(i, hv.kappa[c] + 0.02, f"κ={hv.kappa[c]:.2f}\nP={hv.precision[c]:.2f} R={hv.recall[c]:.2f}", ha="center", fontsize=7)
ax[0].set_title("Coder vs human labels, held-out half (n = 150)")
M1 = E1R.groupby("model")[CATS].mean()
im = ax[1].imshow(M1.values, cmap="Blues", vmin=0, vmax=1); ax[1].set_xticks(range(5)); ax[1].set_xticklabels([short[c] for c in CATS], rotation=20)
ax[1].set_yticks(range(len(M1))); ax[1].set_yticklabels(M1.index)
for i in range(M1.shape[0]):
    for j in range(5): ax[1].text(j, i, f"{M1.values[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if M1.values[i, j] > 0.6 else "black")
ax[1].set_title("Share of explanations mentioning each cue (E1)")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_n11_mentions.png", dpi=130); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 4.6)); mk = {"nu": "o", "eld": "s", "xr": "^", "mel": "D"}
for a, col, t in zip(ax, ["causal_effect", "e1_weight"], ["Causal effect of the edit (E2, minus null floor)", "Standardized weight on E1 choices"]):
    for r in SVU.itertuples():
        v = getattr(r, col)
        if pd.notna(v): a.scatter(r.mention_rate, abs(v) if col == "e1_weight" else v, marker=mk[r.cf_proxy], s=60); a.annotate(f"{r.model} {r.cf_proxy}", (r.mention_rate, abs(v) if col == "e1_weight" else v), fontsize=6, xytext=(3, 3), textcoords="offset points")
    a.set_xlabel("share of E1 explanations that mention the cue"); a.set_ylabel(t if col == "causal_effect" else "|weight|"); a.set_title("Said vs used: " + t); a.axhline(0, c="k", lw=.6)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_n11_said_vs_used.png", dpi=130); plt.show()

if len(CCT):
    fig, ax = plt.subplots(figsize=(11, 4)); cues = ["nu", "eld", "xr", "mel"]; ms = sorted(CCT.model.unique()); wdt = 0.8 / len(ms)
    for i, m in enumerate(ms):
        d = CCT[CCT.model == m].set_index("cf_proxy").reindex(cues); x = np.arange(4) + i * wdt
        ax.bar(x, d.cct, wdt, label=m, yerr=[(d.cct - d.ci_lo).clip(lower=0), (d.ci_hi - d.cct).clip(lower=0)], capsize=2)
    ax.set_xticks(np.arange(4) + 0.4 - wdt / 2); ax.set_xticklabels(["node uniformity", "edge-length variation", "crossings", "mean edge length"]); ax.axhline(0, c="k", lw=.8)
    ax.set_ylabel("CCT (> 0 = faithful)"); ax.set_title("Counterfactual faithfulness of explanations (95 % CI)"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_n11_cct.png", dpi=130); plt.show()

e3 = MR[(MR.exp.isin(["E1", "E3"])) & (MR.variant == "default")]
if len(e3):
    fig, ax = plt.subplots(1, len(e3.model.unique()), figsize=(5.5 * len(e3.model.unique()), 3.8), sharey=True); ax = np.atleast_1d(ax)
    for a, (m, g) in zip(ax, e3.groupby("model")):
        g = g.groupby("prompt_id")[CATS].mean(); g.columns = [short[c] for c in CATS]; g.plot.bar(ax=a, rot=30, legend=a is ax[-1]); a.set_title(m); a.set_xlabel("")
    ax[0].set_ylabel("mention rate"); plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_n11_by_prompt.png", dpi=125); plt.show()

## 9. Report and manifest

In [12]:
hv = VAL[VAL.part.str.startswith("held")]
REP = {"runtime_min": round((time.time() - T0) / 60, 1), "answers": int(len(A)), "with_explanation": int(A.has_reason.sum()),
       "coder_heldout": hv.set_index("category")[["kappa", "precision", "recall", "agreement"]].to_dict("index"),
       "e1_mention_rates": E1R.groupby("model")[CATS].mean().round(3).to_dict("index"),
       "said_vs_used": SVU.round(3).to_dict("records"), "cct": CCT.to_dict("records"), "claim_validity": CV.to_dict("records")}
json.dump(REP, open(f"{OUTPUT_DIR}/n11_report.json", "w"), indent=1, default=str)
A.drop(columns=["reason"]).assign(reason=A.reason).to_parquet(f"{OUTPUT_DIR}/coded_reasons.parquet", index=False)
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
lines = [f"N11 faithfulness · {REP['runtime_min']} min · {REP['answers']} answers · {REP['with_explanation']} explanations", ""]
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = f"{OUTPUT_DIR}/{f}"
    if os.path.isfile(p): lines.append(f"{f:30s} {os.path.getsize(p)/1024:10.1f} KB")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines))

N11 faithfulness · 37.6 min · 82828 answers · 81059 explanations

coded_reasons.parquet              2658.3 KB
coder.py                              1.8 KB
fig_n11_by_prompt.png                59.3 KB
fig_n11_cct.png                      40.9 KB
fig_n11_mentions.png                 85.6 KB
fig_n11_said_vs_used.png             88.4 KB
n11_cct.csv                           1.0 KB
n11_claim_validity.csv                0.4 KB
n11_coder_validation.csv              1.0 KB
n11_mention_rates.csv                 3.7 KB
n11_report.json                       8.8 KB
n11_said_vs_used.csv                  0.8 KB
pdcp_utils.py                        22.4 KB
